# MJO Lag Correlation / Regression
Replicates NCL `lag_corr_CESM3_315_316_lat_time.ncl`.
Observations: TRMM precipitation + ERAI U850.
Supports both lag **correlation** and lag **regression**.

In [28]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.dirname(os.path.abspath('mjo_utils.py')))
import mjo_utils as mjo

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [29]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster

In [30]:
cluster = PBSCluster(
    account="P03010039",
    interface="ext",
    walltime="12:00:00",
    queue="main",
    cores=1,
    memory="8GB",
    processes=4,
    local_directory="/glade/derecho/scratch/rneale/dask-temp",
    log_directory="/glade/derecho/scratch/rneale/dask-logs"
)

cluster.scale(jobs=8)
client = Client(cluster)

client

/glade/work/rneale/conda-envs/neale_vproc2/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39993 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /node/crhtc65.hpc.ucar.edu/48602/proxy/39993/status,
Dashboard: /node/crhtc65.hpc.ucar.edu/48602/proxy/39993/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.173:33767,Workers: 0
Dashboard: /node/crhtc65.hpc.ucar.edu/48602/proxy/39993/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Settings

In [33]:
# ── Data source ───────────────────────────────────────────────
data_src = 'CAM'    # 'OBS' → TRMM precip + ERAI U850
                    # 'CAM' → CAM/CESM daily output

# ── OBS directories (TRMM + ERAI) ────────────────────────────
dir_erai  = '/glade/work/rneale/data/ERAI/'
dir_trmm  = '/glade/work/rneale/data/TRMM/'
dir_fig = '/glade/u/home/rneale/python/python-figs/mjo/'
dir_case = '/glade/derecho/scratch/rneale/archive/'

# ── CAM/CESM settings ─────────────────────────────────────────
#case_cam  = 'b.e30_alpha08o.B1850C_MTso.ne30_t232_wgx3.334'
case_cam = 'f.e22.FHIST.f09_f09.CAM6.L32.000'

dir_cam   = (dir_case+case_cam+'/tseries/')

#precip_vars_cam = ('PRECC', 'PRECL')   # or ('PRECT',) if available
precip_vars_cam = ('PRECT',)

# ── CAM input mode ────────────────────────────────────────────
# 'tseries' : per-variable daily-mean files already exist in dir_cam
#             expects: {dir_cam}/{case_cam}_dmeans_ts_{var}.nc
# 'hist'    : build timeseries from raw history files, save to dir_cam
#             set hist_dir_cam and hist_channel_cam below
cam_data_mode    = 'hist'
hist_dir_cam     = (dir_case+case_cam+'/atm/hist/')

hist_channel_cam = 'h1'          # h0a / h1a / h2a / h0 / h1 etc.

# ── Year range ────────────────────────────────────────────────
# OBS:  TRMM 1998-2009, ERAI from 1979 — overlap used: 2000-2009
# CAM:  model years (e.g. 20-49 for CESM3 316)
year_start = 2000   # OBS
year_end   = 2009   # OBS
year_start_cam = 1980
year_end_cam   = 1999

# ── Analysis type ─────────────────────────────────────────────
analysis = 'corr'    # 'corr'  → lag correlation
                     # 'regr'  → lag regression (per 1-std of base index)

# ── Lanczos bandpass filter ───────────────────────────────────
n_weights = 201      # number of filter weights
fca       = 1./100.  # low  cutoff (cycles/day): 100-day period
fcb       = 1./20.   # high cutoff (cycles/day):  20-day period

# ── Spatial domain to read in ─────────────────────────────────
lat_s_read, lat_n_read = -20., 30.
lon_w_read, lon_e_read =  30., 210.

# ── Base-index (IO2) region ───────────────────────────────────
lat_s_io, lat_n_io = -15., 0.
lon_w_io, lon_e_io =  60., 90.

# ── Lag-lon plot: lat band to average ─────────────────────────
lat_s_ew, lat_n_ew = -15., 0.

# ── Lag-lat plot: lon band to average ─────────────────────────
lon_w_ns, lon_e_ns = 80., 100.

# ── Lag settings ──────────────────────────────────────────────
mxlag   = 25
seasons = ['winter', 'summer']   # 'winter'=DJF, 'summer'=JJA

# ── Plot settings ─────────────────────────────────────────────
case_label = case_cam if data_src == 'CAM' else 'TRMM-ERAI'

# Contour levels for correlation
clevs_corr = 2.0*np.array([0.04,0.08,0.12,0.16,0.20,0.24,0.28,0.32,0.36,0.40,0.50])
clevs_corr = np.concatenate([-clevs_corr[::-1], clevs_corr])

# Contour levels for regression
clevs_regr_p = np.linspace(-4.0, 4.0, 21)
clevs_regr_u = np.linspace(-3.0, 3.0, 21)

# Reference lon/lat lines (IO2 region edges)
ref_lons = [lon_w_io, lon_e_io]
ref_lats = [lat_s_io, lat_n_io]

# Maritime Continent shading on EW plots
mc_box = (115., 145.)

print(f'Settings loaded.  data_src={data_src}  analysis={analysis}')
if data_src == 'CAM':
    mode_str = cam_data_mode
    if cam_data_mode == 'hist':
        mode_str += f'  channel={hist_channel_cam}'
    print(f'  CAM mode  : {mode_str}')
    print(f'  Years     : {year_start_cam}–{year_end_cam}')
else:
    print(f'  Years     : {year_start}–{year_end}')
print(f'  Max lag   : ±{mxlag} days  |  Seasons: {seasons}')

Settings loaded.  data_src=CAM  analysis=corr
  CAM mode  : hist  channel=h1
  Years     : 1980–1999
  Max lag   : ±25 days  |  Seasons: ['winter', 'summer']


## Load Data

In [ ]:
if data_src == 'OBS':
    # ── TRMM precipitation (mm/day) ───────────────────────────────────────
    print('Loading TRMM ...')
    ds_trmm = xr.open_dataset(dir_trmm + '3B42.1998-2009.1x1.v6.nc',
                              decode_timedelta=False)
    times_trmm = pd.DatetimeIndex(ds_trmm['time'].values)

    lat_t = ds_trmm['lat'].values.astype(float)
    lon_t = ds_trmm['lon'].values.astype(float)
    lat_mask_t = (lat_t >= lat_s_read) & (lat_t <= lat_n_read)
    lon_mask_t = (lon_t >= lon_w_read) & (lon_t <= lon_e_read)

    prect_raw = ds_trmm['precip'].values[:, lat_mask_t, :][:, :, lon_mask_t]
    lat_t = lat_t[lat_mask_t]
    lon_t = lon_t[lon_mask_t]

    ymask_t = (times_trmm.year >= year_start) & (times_trmm.year <= year_end)
    prect_raw = prect_raw[ymask_t]
    times_trmm = times_trmm[ymask_t]
    print(f'  TRMM  : {prect_raw.shape}  {times_trmm[0].date()} – {times_trmm[-1].date()}')

    # ── ERAI U850 (m/s) ──────────────────────────────────────────────────
    print('Loading ERAI U850 ...')
    ds_u850 = xr.open_dataset(dir_erai + 'u850.day.mean.nc', decode_times=False)
    times_erai = mjo.decode_julian_day(ds_u850['time'])

    lat_e = ds_u850['lat'].values.astype(float)
    lon_e = ds_u850['lon'].values.astype(float)
    if lat_e[0] > lat_e[-1]:
        lat_e = lat_e[::-1]
        u850_all = ds_u850['u850'].values[:, ::-1, :]
    else:
        u850_all = ds_u850['u850'].values

    lat_mask_e = (lat_e >= lat_s_read) & (lat_e <= lat_n_read)
    lon_mask_e = (lon_e >= lon_w_read) & (lon_e <= lon_e_read)
    u850_raw = u850_all[:, lat_mask_e, :][:, :, lon_mask_e]
    lat_e = lat_e[lat_mask_e]
    lon_e = lon_e[lon_mask_e]

    ymask_e = (times_erai.year >= year_start) & (times_erai.year <= year_end)
    u850_raw = u850_raw[ymask_e]
    times_erai = times_erai[ymask_e]
    print(f'  ERAI  : {u850_raw.shape}  {times_erai[0].date()} – {times_erai[-1].date()}')

    # ── Align time axes ───────────────────────────────────────────────────
    dates_t = pd.DatetimeIndex([d.date() for d in times_trmm])
    dates_e = pd.DatetimeIndex([d.date() for d in times_erai])
    common  = dates_t.intersection(dates_e)
    prect_raw = prect_raw[np.isin(dates_t, common)]
    u850_raw  = u850_raw[np.isin(dates_e, common)]
    times     = times_trmm[np.isin(dates_t, common)]

    # Coordinate arrays for each variable (different grids)
    lat_p, lon_p = lat_t, lon_t   # TRMM grid for precip
    lat_u, lon_u = lat_e, lon_e   # ERAI grid for U850

elif data_src == 'CAM':
    # ── CAM/CESM daily output ─────────────────────────────────────────────
    # cam_data_mode='tseries': load pre-existing per-variable daily-mean files
    # cam_data_mode='hist'   : build timeseries from raw history files first
    print(f'Loading CAM ({cam_data_mode}): {case_cam} ...')
    prect_raw, u850_raw, lat_c, lon_c, times = mjo.load_cam_daily(
        data_dir         = dir_cam,
        case_name        = case_cam,
        year_start       = year_start_cam,
        year_end         = year_end_cam,
        lat_s            = lat_s_read,
        lat_n            = lat_n_read,
        lon_w            = lon_w_read,
        lon_e            = lon_e_read,
        precip_vars      = precip_vars_cam,
        hist_dir         = hist_dir_cam if cam_data_mode == 'hist' else None,
        channel          = hist_channel_cam,
    )
    # Both variables on the same CAM grid
    lat_p, lon_p = lat_c, lon_c
    lat_u, lon_u = lat_c, lon_c
    print(f'  PRECT : {prect_raw.shape}  {times[0].date()} – {times[-1].date()}')
    print(f'  U850  : {u850_raw.shape}')

else:
    raise ValueError(f"Unknown data_src '{data_src}'. Use 'OBS' or 'CAM'.")

print(f'\nDone. {len(times)} days loaded.')
print(f'  Precip grid: lat {lat_p[0]:.1f}–{lat_p[-1]:.1f} ({len(lat_p)} pts), '
      f'lon {lon_p[0]:.1f}–{lon_p[-1]:.1f} ({len(lon_p)} pts)')
print(f'  U850   grid: lat {lat_u[0]:.1f}–{lat_u[-1]:.1f} ({len(lat_u)} pts), '
      f'lon {lon_u[0]:.1f}–{lon_u[-1]:.1f} ({len(lon_u)} pts)')

Loading CAM (hist): f.e22.FHIST.f09_f09.CAM6.L32.000 ...
Building timeseries from hist (h1) ...
  Found 730 h1 files  (f.e22.FHIST.f09_f09.CAM6.L32.000.cam.h1.1980-01-06-00000.nc … f.e22.FHIST.f09_f09.CAM6.L32.000.cam.h1.1999-12-27-00000.nc)
  Writing → /glade/derecho/scratch/rneale/archive/f.e22.FHIST.f09_f09.CAM6.L32.000/tseries/f.e22.FHIST.f09_f09.CAM6.L32.000_dmeans_ts_PRECT.nc


## Bandpass Filter & Compute Lag Series

In [ ]:
print('Computing base index (IO2 region, precip) ...')
base_idx = mjo.compute_base_index(
    prect_raw, lat_p, lon_p,
    lat_s=lat_s_io, lat_n=lat_n_io,
    lon_w=lon_w_io, lon_e=lon_e_io,
    n_weights=n_weights, fca=fca, fcb=fcb
)
print(f'  Base index std (non-NaN): {np.nanstd(base_idx):.3f} mm/day')

# ── Lag-lon time series: lat-averaged ────────────────────────────────────
print('Computing lat-averaged time-lon series ...')
P_timeLon = mjo.compute_time_lon_series(
    prect_raw, lat_p, lat_s=lat_s_ew, lat_n=lat_n_ew,
    n_weights=n_weights, fca=fca, fcb=fcb
)   # (time, lon_p)

U_timeLon = mjo.compute_time_lon_series(
    u850_raw, lat_u, lat_s=lat_s_ew, lat_n=lat_n_ew,
    n_weights=n_weights, fca=fca, fcb=fcb
)   # (time, lon_u)

# ── Lag-lat time series: lon-averaged ────────────────────────────────────
print('Computing lon-averaged time-lat series ...')
P_timeLat = mjo.compute_time_lat_series(
    prect_raw, lon_p, lon_w=lon_w_ns, lon_e=lon_e_ns,
    n_weights=n_weights, fca=fca, fcb=fcb
)   # (time, lat_p)

U_timeLat = mjo.compute_time_lat_series(
    u850_raw, lon_u, lon_w=lon_w_ns, lon_e=lon_e_ns,
    n_weights=n_weights, fca=fca, fcb=fcb
)   # (time, lat_u)

print('Done.')

## Compute Lag Correlations / Regressions

In [ ]:
# Choose function based on analysis setting
lag_fn = mjo.mjo_lag_corr if analysis == 'corr' else mjo.mjo_lag_regr

results = {}   # keyed by season

for season in seasons:
    print(f'Computing lag {analysis} — {season} ...')

    rp_lon, lags = lag_fn(base_idx, P_timeLon, times, mxlag=mxlag, season=season)
    ru_lon, _    = lag_fn(base_idx, U_timeLon, times, mxlag=mxlag, season=season)
    rp_lat, _    = lag_fn(base_idx, P_timeLat, times, mxlag=mxlag, season=season)
    ru_lat, _    = lag_fn(base_idx, U_timeLat, times, mxlag=mxlag, season=season)

    results[season] = dict(
        P={'lon': rp_lon, 'lat': rp_lat},
        U={'lon': ru_lon, 'lat': ru_lat},
        lags=lags
    )
    print(f'  PRECT lon  min/max: {np.nanmin(rp_lon):.3f} / {np.nanmax(rp_lon):.3f}')
    print(f'  U850  lon  min/max: {np.nanmin(ru_lon):.3f} / {np.nanmax(ru_lon):.3f}')

print('\nAll seasons done.')

## Plots

In [ ]:
# ── Contour levels ─────────────────────────────────────────────────────────
if analysis == 'corr':
    clevs_p = clevs_corr
    clevs_u = clevs_corr
    cbar_lbl_p = 'Correlation'
    cbar_lbl_u = 'Correlation'
else:
    clevs_p = clevs_regr_p
    clevs_u = clevs_regr_u
    cbar_lbl_p = 'mm/day per std(base)'
    cbar_lbl_u = 'm/s per std(base)'

# ── Winter (DJF) ──────────────────────────────────────────────────────────
season = 'winter'
res = results[season]

fig, axes = mjo.make_lag_panel(
    r_prect      = res['P'],
    r_u850       = res['U'],
    lags         = res['lags'],
    lon_coords   = lon_p,
    lat_coords   = lat_p,
    lon_coords_u = lon_u,
    lat_coords_u = lat_u,
    season_label = 'Winter (DJF)',
    case_label   = case_label,
    clevs_p      = clevs_p,
    clevs_u      = clevs_u,
    lon_lim      = (lon_w_read, lon_e_read),
    lat_lim      = (lat_s_read, lat_n_read),
    ref_lon      = ref_lons,
    ref_lat      = ref_lats,
    mc_box       = mc_box,
    analysis_label = analysis.capitalize(),
    cbar_label_p   = cbar_lbl_p,
    cbar_label_u   = cbar_lbl_u,
)
fname = case_label+'_'+analysis+'_'+season+'.png'
fig.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summer (JJA) ──────────────────────────────────────────────────────────
season = 'summer'
res = results[season]

fig, axes = mjo.make_lag_panel(
    r_prect      = res['P'],
    r_u850       = res['U'],
    lags         = res['lags'],
    lon_coords   = lon_p,
    lat_coords   = lat_p,
    lon_coords_u = lon_u,
    lat_coords_u = lat_u,
    season_label = 'Summer (JJA)',
    case_label   = case_label,
    clevs_p      = clevs_p,
    clevs_u      = clevs_u,
    lon_lim      = (lon_w_read, lon_e_read),
    lat_lim      = (lat_s_read, lat_n_read),
    ref_lon      = ref_lons,
    ref_lat      = ref_lats,
    mc_box       = mc_box,
    analysis_label = analysis.capitalize(),
    cbar_label_p   = cbar_lbl_p,
    cbar_label_u   = cbar_lbl_u,
)
fname = case_label+'_'+analysis+'_'+season+'.png'
plt.show()